# Accuracy against the survey center

The accuracy term `S_acc = 1 - E` lives on a bounded scale where a prediction
that ignores demographics already scores high. This notebook scores that
prediction. It takes the equal-cell survey center over a run's retained cells,

$$\bar{s} = \frac{1}{n}\sum_i s_i,$$

predicts it for every cell, $m_i^{(0)} = \bar{s}$, and scores it exactly as a
model is scored:

$$S_{\mathrm{acc}}^{(0)} = 1 - \frac{1}{n}\sum_i \mathrm{nEMD}(s_i, \bar{s}).$$

How far a model's accuracy moves from that floor toward a perfect score is

$$S_{\mathrm{acc}}^{\mathrm{above\ null}} = \frac{S_{\mathrm{acc}} - S_{\mathrm{acc}}^{(0)}}{1 - S_{\mathrm{acc}}^{(0)}},$$

zero for a model no better than the survey center, one for perfect cell-level
prediction, negative for a model that does worse.

The survey center is scored by `group_fidelity` itself, so its other terms come
from the code that scores every model: its center alignment is 1 by
construction, its pairwise distances are all zero, so `A = 0`, the flatness
rule sets structure to 0, and its PFS is 0. A leave-one-out center, which drops
each cell from its own prediction, rides beside it to show that using a cell's
own survey answers inside $\bar{s}$ changes nothing that matters.

Only the runs the manuscript reports are scored: the first replicate of the six
open models and GPT-5.6-Terra, as released and under the German and
Mexican-Spanish variants, plus the two Mixtral references, which is the 164
pooled combinations. The pooled Spanish variant and later replicates are left
out even when their outputs are on disk.

In [1]:
from culture.fidelity import GROUPS_TABLE
from culture.fidelity_baseline import (
    BASELINE_SUMMARY_TABLE,
    BASELINE_TABLE,
    baseline_summary,
    build_baseline,
    check_baseline,
    manuscript_runs,
)
from culture.tables import read_csv
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
runs = manuscript_runs()
baseline = build_baseline(runs=runs)
print(len(baseline), "rows over", (baseline["group"] == "population").sum(), "pooled combinations")

5084 rows over 164 pooled combinations


## Reconciling against the groups table

Every row scores the model's accuracy again on the same cells. The check
asserts that it equals `population_fidelity_groups.csv` row for row, and that
wherever a level holds two cells or more the survey center is flat, has
`A = 0`, never scores a positive PFS, and sits exactly on the survey center.

In [3]:
groups = read_csv(GROUPS_TABLE)
assert groups is not None
check_baseline(baseline, groups, runs)
print(len(baseline), "rows match the groups table")

5084 rows match the groups table


## The tables

`population_fidelity_baseline.csv` holds one row per run, mode, group and
level. `population_fidelity_baseline_summary.csv` summarises three scopes by
question and mode, each closed by an `all` row: `population`, the pooled
combinations; `subgroups`, every demographic level of the sweep's conditions,
with the Mixtral references left out as the subgroup analysis leaves them out;
and `home`, the German variant inside Germany and the Mexican-Spanish variant
inside Mexico, each read against its own country's center.

In [4]:
summary = baseline_summary(baseline)
write_csv(baseline, BASELINE_TABLE)
write_csv(summary, BASELINE_SUMMARY_TABLE)
pooled = summary.query("scope == 'population' and question == 'all'").iloc[0]
print(pooled["n_above_null"], "of", pooled["n_scores"], "pooled combinations beat the survey center")

7 of 164 pooled combinations beat the survey center


## Reading the table

Accuracy is high on this scale because the survey center is already close to
every cell, and almost no combination beats it. A model's cell-level accuracy
therefore comes mostly from landing near the survey center, which is the
machine-bias diagnosis read off the accuracy term.

Within a question the survey center moves by thousandths across runs, so the
above-null score is a rescaling of accuracy that changes no ranking; it only
moves the zero to the point where demographic conditioning starts to help. It
has no lower bound and stays out of PFS, which already scores the survey center
at 0 through adaptability and structure.